FAST — NUCES, Karachi Campus

<div style="border-bottom:3px solid #1F3864;padding-bottom:10px">
<h1 style="color:#1F3864;margin-bottom:2px">Lab 07 — NumPy and the N-Dimensional Array</h1>

Department of Artificial Intelligence &amp; Data Science<br>

<i>Prepared by Khadeejah Ashraf, Lab Instructor</i>
</div>

Read the explanation, run the code cell below it with **Shift + Enter**, and compare what you
get with what you expected. Cells marked **Your turn** are for you to fill in.

**Contents**

1. Why NumPy — less memory, far more speed
2. Creating arrays
3. The anatomy of an ndarray — shape, dtype, overflow
4. Indexing and slicing · views vs copies
5. Boolean masks and fancy indexing
6. Reshaping and rearranging
7. Broadcasting
8. Vectorization, ufuncs and the `axis` argument
9. Random numbers
10. Linear algebra essentials
11. Two worked examples
12. Class Tasks (C1 – C5)
13. Home Tasks

In [1]:
import numpy as np
import sys, time

print("NumPy version:", np.__version__)
np.set_printoptions(precision=3, suppress=True)   # tidier printing for this notebook

NumPy version: 2.1.3


---
## 1. Why NumPy

A Python list stores *pointers* to full Python objects scattered in memory. A NumPy array
stores the numbers themselves, all of one type, in one contiguous block.

In [2]:
py_list = list(range(1000))
np_arr  = np.arange(1000)

print("list :", sys.getsizeof(py_list) + sum(sys.getsizeof(v) for v in py_list), "bytes")
print("array:", np_arr.nbytes, "bytes")

list : 36056 bytes
array: 8000 bytes


In [3]:
# the same calculation, two ways, on one million elements
n = 1_000_000
py  = list(range(n))
arr = np.arange(n, dtype=np.float64)

t0 = time.perf_counter()
out = []
for value in py:                       # the Python way
    out.append(value * 2.0 + 1.0)
t_loop = time.perf_counter() - t0

t0 = time.perf_counter()
result = arr * 2.0 + 1.0               # the NumPy way - one expression
t_np = time.perf_counter() - t0

print("loop  :", round(t_loop * 1000, 1), "ms")
print("numpy :", round(t_np * 1000, 1), "ms")
print("faster by about", round(t_loop / t_np), "times")
print("same answer?", np.allclose(out, result))

loop  : 90.1 ms
numpy : 6.0 ms
faster by about 15 times
same answer? True


In [4]:
# a first taste - notice that no loop appears anywhere
marks = np.array([78, 45, 90, 62])

print(marks * 2)              # every element doubled
print(marks + 5)              # five added to every element
print(marks.mean(), marks.max())
print(marks[marks >= 50])     # only the passing marks

[156  90 180 124]
[83 50 95 67]
68.75 90
[78 90 62]


In [5]:
# the same expressions on a LIST behave completely differently
marks_list = [78, 45, 90, 62]
print(marks_list * 2)         # the list is repeated, not doubled

try:
    print(marks_list + 5)
except TypeError as err:
    print("TypeError:", err)

[78, 45, 90, 62, 78, 45, 90, 62]
TypeError: can only concatenate list (not "int") to list


---
## 2. Creating Arrays

| Name | ndim | Shape | Example |
|---|---|---|---|
| Vector | 1 | `(n,)` | one student's marks |
| Matrix | 2 | `(rows, cols)` | a class's marks table |
| Tensor | 3+ | `(a, b, c)` | a colour image: h × w × 3 |

In [6]:
a = np.array([1, 2, 3, 4])                             # vector
b = np.array([[1, 2, 3], [4, 5, 6]])                   # matrix
c = np.array([[[1, 2], [3, 4]], [[5, 6], [7, 8]]])     # tensor

print(a.shape, b.shape, c.shape)
print(a.ndim, b.ndim, c.ndim)

(4,) (2, 3) (2, 2, 2)
1 2 3


In [7]:
print(np.arange(5))                 # 0 to 4
print(np.arange(2, 11, 2))          # even numbers, 11 excluded
print(np.linspace(0, 1, 5))         # five values, both ends included
print(np.arange(0.1, 0.4, 0.1))     # careful! see below

[0 1 2 3 4]
[ 2  4  6  8 10]
[0.   0.25 0.5  0.75 1.  ]
[0.1 0.2 0.3 0.4]


That last line asked for values *below* 0.4 and got four of them — including 0.4. Floating-point
steps do not land exactly where you expect, so **whenever the step is fractional, use
`linspace`**.

In [8]:
print(np.zeros((2, 3)))          # floats by default
print(np.ones(4, dtype=int))
print(np.full((2, 2), 7))
print(np.eye(3, dtype=int))      # the identity matrix - never hard-code this

[[0. 0. 0.]
 [0. 0. 0.]]
[1 1 1 1]
[[7 7]
 [7 7]]
[[1 0 0]
 [0 1 0]
 [0 0 1]]


In [9]:
base = np.array([[1, 2, 3], [4, 5, 6]])

print(np.zeros_like(base))       # same shape and dtype as base
print(np.ones_like(base).shape)

[[0 0 0]
 [0 0 0]]
(2, 3)


**Your turn.** Without typing out any values, build: the multiples of 5 from 5 to 50 inclusive;
a 4 × 4 matrix of sevens; and six evenly spaced values between −1 and 1. Print each with its
shape.

In [10]:
# Your code here

---
## 3. The Anatomy of an ndarray

When an operation misbehaves, print these first — nine times out of ten the shape is not what
you assumed.

In [11]:
a = np.array([[10, 20, 30, 40],
              [50, 60, 70, 80],
              [90, 100, 110, 120]])

print("shape   :", a.shape)      # a tuple, one entry per dimension
print("ndim    :", a.ndim)
print("size    :", a.size)
print("dtype   :", a.dtype)
print("itemsize:", a.itemsize, "bytes")
print("nbytes  :", a.nbytes, "bytes")

shape   : (3, 4)
ndim    : 2
size    : 12
dtype   : int64
itemsize: 8 bytes
nbytes  : 96 bytes


**Axes, once and for all:** `axis=0` runs *down* the rows, `axis=1` runs *across* the columns.
The shape tuple is in the same order, so in `(3, 4)` the `3` is the length of axis 0.

In [12]:
print(np.array([1, 2, 3]).dtype)          # all integers
print(np.array([1.0, 2, 3]).dtype)        # one float makes them all float
print(np.array([1, 2, "three"]).dtype)    # one string makes them all strings
print(np.array([1, 2, 3], dtype=np.float32).dtype)

ints = np.array([1, 2, 3])
ints[0] = 9.9                             # stored into an integer array
print(ints)                               # the fraction is silently dropped

int64
float64
<U21
float32
[9 2 3]


In [13]:
# integer overflow: NumPy wraps around silently, unlike plain Python
small = np.array([120, 125], dtype=np.int8)     # int8 holds -128 to 127
print(small + 10)

print(np.array([1, 2, 3]) / np.array([2, 2, 2]))    # division always gives floats

[-126 -121]
[0.5 1.  1.5]


---
## 4. Indexing and Slicing

One dimension behaves exactly like a list. Two or more dimensions use **one** pair of brackets
with commas: `a[row, column]`.

In [14]:
v = np.arange(10, 60, 10)
print(v)
print(v[0], v[-1])
print(v[1:4])
print(v[::2])
print(v[::-1])

[10 20 30 40 50]
10 50
[20 30 40]
[10 30 50]
[50 40 30 20 10]


In [15]:
a = np.arange(20).reshape(4, 5)
print(a)
print(a[1, 3])        # one element: row 1, column 3
print(a[1])           # a whole row
print(a[:, 2])        # a whole column - every row, column 2
print(a[1:3, 1:4])    # a rectangular block
print(a[-1, -1])      # the bottom-right corner

[[ 0  1  2  3  4]
 [ 5  6  7  8  9]
 [10 11 12 13 14]
 [15 16 17 18 19]]
8
[5 6 7 8 9]
[ 2  7 12 17]
[[ 6  7  8]
 [11 12 13]]
19


Write `a[1, 3]`, **not** `a[1][3]`. Both give the same element, but `a[1][3]` builds a whole
temporary row first — and only the comma form accepts slices on both axes at once.

In [16]:
t = np.arange(12).reshape(2, 2, 3)
print(t)
print(t[0, 1, 2])      # block 0, row 1, element 2
print(t.shape)

[[[ 0  1  2]
  [ 3  4  5]]

 [[ 6  7  8]
  [ 9 10 11]]]
5
(2, 2, 3)


### 4.1 Views and copies — the section most often skipped, most often regretted

**Slicing an array does not copy it.** The slice is a *view*: a second window onto the same
memory.

In [17]:
a = np.arange(10)
s = a[2:5]          # a view, not a copy
s[0] = 999

print(a)            # the ORIGINAL has changed
print(s.base is a)  # the view remembers what it looks at

[  0   1 999   3   4   5   6   7   8   9]
True


In [18]:
a = np.arange(10)
s = a[2:5].copy()   # an independent array
s[0] = 999

print(a)            # untouched
print(s)

[0 1 2 3 4 5 6 7 8 9]
[999   3   4]


| Operation | Gives you |
|---|---|
| `a[2:5]`, `a[:, 1]`, `a.T`, `a.reshape(...)`, `a.ravel()` | a **view** — shares memory |
| `a[a > 5]` (boolean mask) | a **copy** |
| `a[[0, 2, 4]]` (fancy indexing) | a **copy** |
| `a.copy()`, `a.flatten()`, `a * 2` | a **copy** |

If an array you never assigned to has mysteriously changed, look for a slice of it you wrote to
earlier.

**Your turn.** From `np.arange(36).reshape(6, 6)`, print: the element at row 2 column 4; the
last row; the third column; the middle 2 × 2 block; and every second row.

In [19]:
m = np.arange(36).reshape(6, 6)
# Your code here

---
## 5. Boolean Masks and Fancy Indexing

Comparing an array with a value gives an array of `True`/`False`, which can then be used as an
index. This replaces the filtering loop entirely.

In [20]:
marks = np.array([78, 45, 90, 62, 33, 88])

mask = marks >= 50
print(mask)
print(marks[mask])                            # select using the mask
print(marks[(marks >= 50) & (marks < 85)])    # two conditions
print(mask.sum(), "passed out of", marks.size)
print(np.any(marks > 85), np.all(marks > 30))

[ True False  True  True False  True]
[78 90 62 88]
[78 62]
4 passed out of 6
True True


In [21]:
# why you must use & | ~ and not and / or / not
try:
    marks[(marks >= 50) and (marks < 85)]
except ValueError as err:
    print("ValueError:", err)

ValueError: The truth value of an array with more than one element is ambiguous. Use a.any() or a.all()


In [22]:
# a mask on the LEFT of an assignment changes only those elements
marks = np.array([78, 45, 90, 62, 33, 88])
marks[marks < 50] = 50          # lift every failing mark to 50
print(marks)

[78 50 90 62 50 88]


In [23]:
v = np.array([10, 20, 30, 40, 50])
print(v[[0, 2, 4]])         # pick three positions
print(v[[4, 4, 0]])         # order and repetition are allowed

a = np.arange(20).reshape(4, 5)
print(a[[0, 2], [1, 3]])    # the elements (0,1) and (2,3)
print(a[[0, 2]])            # rows 0 and 2, whole

[10 30 50]
[50 50 10]
[ 1 13]
[[ 0  1  2  3  4]
 [10 11 12 13 14]]


In [24]:
marks = np.array([78, 45, 90, 62])

print(np.where(marks >= 50, "pass", "fail"))
print(np.where(marks >= 50, marks, 0))      # keep, or replace with zero
print(np.where(marks >= 50)[0])             # the positions themselves

['pass' 'fail' 'pass' 'pass']
[78  0 90 62]
[0 2 3]


**Your turn.** From the array below print: every value above the mean; how many values are
divisible by 3; the array with every negative value replaced by 0; and the positions of the
three largest values (hint: `np.argsort`).

In [25]:
data = np.array([12, -7, 40, 3, -25, 18, 33, -2, 27])
# Your code here

---
## 6. Reshaping and Rearranging

In [26]:
a = np.arange(12)
print(a.reshape(3, 4))
print(a.reshape(2, 6).shape)
print(a.reshape(3, -1).shape)     # -1 means "whatever fits"
print(a.reshape(2, 2, 3).shape)

[[ 0  1  2  3]
 [ 4  5  6  7]
 [ 8  9 10 11]]
(2, 6)
(3, 4)
(2, 2, 3)


In [27]:
a = np.arange(6).reshape(2, 3)
print(a.ravel())        # a view where possible
print(a.flatten())      # always a copy

r = a.ravel()
r[0] = 99               # writing through the view...
print(a)                # ...changes the original

[0 1 2 3 4 5]
[0 1 2 3 4 5]
[[99  1  2]
 [ 3  4  5]]


In [28]:
a = np.arange(6).reshape(2, 3)
print(a.T)
print(a.T.shape)

v = np.array([1, 2, 3])
print(v.shape, v[:, np.newaxis].shape, v[np.newaxis, :].shape)
print(v[:, np.newaxis])

[[0 3]
 [1 4]
 [2 5]]
(3, 2)
(3,) (3, 1) (1, 3)
[[1]
 [2]
 [3]]


**A vector is neither a row nor a column.** Shape `(3,)` has only one axis — it is not `(1, 3)`
and not `(3, 1)`. Most shape complaints come from assuming otherwise, and `newaxis` is the fix.

In [29]:
a = np.array([[1, 2], [3, 4]])
b = np.array([[5, 6], [7, 8]])

print(np.concatenate([a, b], axis=0))     # stack rows
print(np.concatenate([a, b], axis=1))     # stack columns
print(np.vstack([a, b]).shape, np.hstack([a, b]).shape)
print(np.stack([a, b]).shape)             # a NEW axis: (2, 2, 2)
print(np.split(np.arange(9), 3))

[[1 2]
 [3 4]
 [5 6]
 [7 8]]
[[1 2 5 6]
 [3 4 7 8]]
(4, 2) (2, 4)
(2, 2, 2)
[array([0, 1, 2]), array([3, 4, 5]), array([6, 7, 8])]


---
## 7. Broadcasting

NumPy compares the two shapes **from the right-hand end**, one axis at a time:

* equal lengths → fine
* one of them is `1` → that operand is stretched
* otherwise → `ValueError`

A shape with fewer axes is treated as though it had `1`s on the left.

```
a.shape      (3, 1)
b.shape         (4,)   ->  treated as (1, 4)
--------------------
compare       1 vs 4   ->  stretch a to 4
              3 vs 1   ->  stretch b to 3
result       (3, 4)
```

In [30]:
a = np.array([[1, 2, 3], [4, 5, 6]])

print(a * 10)            # a scalar broadcasts to everything
print(a - a.mean())      # centre the whole array on zero

[[10 20 30]
 [40 50 60]]
[[-2.5 -1.5 -0.5]
 [ 0.5  1.5  2.5]]


In [31]:
col = np.array([[10], [20], [30]])     # shape (3, 1)
row = np.array([1, 2, 3, 4])           # shape (4,)

print(col.shape, row.shape)
print(col + row)                       # -> shape (3, 4)

(3, 1) (4,)
[[11 12 13 14]
 [21 22 23 24]
 [31 32 33 34]]


In [32]:
# the genuinely useful case: one value per column, across a whole table
marks   = np.array([[78, 45, 90],
                    [88, 92, 95],
                    [40, 38, 52]])
weights = np.array([0.3, 0.3, 0.4])        # one weight per subject

print(marks * weights)                      # (3,3) * (3,) -> per column
print((marks * weights).sum(axis=1))        # each student's weighted total
print(marks - marks.mean(axis=0))           # centre each subject

[[23.4 13.5 36. ]
 [26.4 27.6 38. ]
 [12.  11.4 20.8]]
[72.9 92.  44.2]
[[  9.333 -13.333  11.   ]
 [ 19.333  33.667  16.   ]
 [-28.667 -20.333 -27.   ]]


In [33]:
# when broadcasting refuses - read the shapes in the message
a = np.ones((3, 4))
b = np.ones((3,))

try:
    print((a + b).shape)        # (3,4) vs (3,) -> compares 4 with 3
except ValueError as err:
    print("ValueError:", err)

try:
    a + np.ones((4, 3))
except ValueError as err:
    print("ValueError:", err)

print((a + b[:, np.newaxis]).shape)   # (3,4) vs (3,1) -> works

ValueError: operands could not be broadcast together with shapes (3,4) (3,) 
ValueError: operands could not be broadcast together with shapes (3,4) (4,3) 
(3, 4)


**Your turn.** Predict the result shape *before* running, then check: `(5,3)` with `(3,)`;
`(5,3)` with `(5,)`; `(5,3)` with `(5,1)`; `(2,1,4)` with `(3,4)`. Repair any failing pair with
`np.newaxis`.

In [34]:
# Your code here

---
## 8. Vectorization, ufuncs and the `axis` Argument

A **ufunc** applies element by element to a whole array and returns a new array.

In [35]:
a = np.array([1, 4, 9, 16])

print(np.sqrt(a))
print(np.exp([0, 1]))
print(np.round(np.log(a), 3))
print(np.abs([-3, 3]))
print(np.maximum([1, 5, 3], [4, 2, 3]))    # element-by-element larger

[1. 2. 3. 4.]
[1.    2.718]
[0.    1.386 2.197 2.773]
[3 3]
[4 5 3]


### 8.1 Turning a loop into an expression

| Instead of a loop that… | Write |
|---|---|
| applies a formula to each element | `result = 2 * a + 1` |
| keeps the elements passing a test | `result = a[a > 50]` |
| chooses between two values per element | `result = np.where(a > 50, a, 0)` |
| adds up a running total | `total = a.sum()` |
| compares two lists position by position | `result = a * b` or `a @ b` |

In [36]:
# the same job written both ways - check they agree
rng = np.random.default_rng(0)
x = rng.random(100_000)

loop_out = []
for value in x:
    loop_out.append(value ** 2 + 3 * value if value > 0.5 else 0.0)

vec_out = np.where(x > 0.5, x ** 2 + 3 * x, 0.0)

print(np.allclose(loop_out, vec_out))

True


### 8.2 Aggregation — the named axis disappears

In [37]:
a = np.array([[1, 2, 3], [4, 5, 6]])

print(a.sum(), a.mean(), a.max())    # the whole array
print(a.sum(axis=0))                 # down the rows  -> one per column
print(a.sum(axis=1))                 # across columns -> one per row
print(a.mean(axis=1, keepdims=True).shape)

print(a.argmax())                    # position in the flattened array
row, col = np.unravel_index(a.argmax(), a.shape)
print("largest value is at row", row, "column", col)

21 3.5 6
[5 7 9]
[ 6 15]
(2, 1)
5
largest value is at row 1 column 2


**What `keepdims` is for:** `a.mean(axis=1)` has shape `(2,)`, which will *not* broadcast back
against `(2, 3)`. With `keepdims=True` it is `(2, 1)`, which broadcasts perfectly — exactly what
row-wise centring needs.

In [38]:
a = np.array([[1, 2, 3], [4, 5, 6]])

try:
    print(a - a.mean(axis=1))                 # (2,3) vs (2,) -> fails
except ValueError as err:
    print("ValueError:", err)

print(a - a.mean(axis=1, keepdims=True))      # (2,3) vs (2,1) -> works

ValueError: operands could not be broadcast together with shapes (2,3) (2,) 
[[-1.  0.  1.]
 [-1.  0.  1.]]


In [39]:
data = np.array([12, 7, 40, 3, 25, 18])

print(data.mean(), data.std().round(3), data.var().round(3))
print(np.median(data), data.min(), data.max())
print(np.sort(data))        # a sorted copy
print(np.argsort(data))     # the positions that would sort it
print(np.cumsum(data))      # running total

17.5 12.339 152.25
15.0 3 40
[ 3  7 12 18 25 40]
[3 1 0 5 4 2]
[ 12  19  59  62  87 105]


---
## 9. Random Numbers

Modern NumPy asks you to create a **generator** first. The seed makes an experiment repeatable —
set one in every experiment you report.

In [40]:
rng = np.random.default_rng(seed=42)

print(rng.integers(1, 7, size=5))                      # five dice rolls
print(rng.random(3).round(3))                          # uniform 0 to 1
print(rng.normal(loc=70, scale=10, size=4).round(2))   # a bell curve
print(rng.choice([2, 5, 9, 10], size=(2, 2)))          # pick from a set

[1 5 4 3 3]
[0.697 0.094 0.976]
[71.28 66.84 69.83 61.47]
[[10  9]
 [ 5  2]]


In [41]:
a = np.random.default_rng(0).integers(0, 100, 5)
b = np.random.default_rng(0).integers(0, 100, 5)

print(a)
print(b)
print(np.array_equal(a, b))     # the same seed always gives the same sequence

[85 63 51 26 30]
[85 63 51 26 30]
True


In [42]:
rng = np.random.default_rng(7)

deck = np.arange(10)
rng.shuffle(deck)          # shuffles in place
print(deck)
print(rng.permutation(5))  # returns a shuffled copy

[8 0 7 1 3 6 2 4 5 9]
[4 3 1 2 0]


---
## 10. Linear Algebra Essentials

`*` multiplies **element by element**. `@` is the **matrix product**. They are different
operations.

In [43]:
A = np.array([[1, 2], [3, 4]])
B = np.array([[5, 6], [7, 8]])

print(A * B)        # element by element
print(A @ B)        # matrix product
print(np.dot(A, B)) # the same as @

M = np.arange(6).reshape(2, 3)
N = np.arange(6).reshape(3, 2)
print((M @ N).shape)      # (2,3) @ (3,2) -> (2,2)

[[ 5 12]
 [21 32]]
[[19 22]
 [43 50]]
[[19 22]
 [43 50]]
(2, 2)


In [44]:
# the rule is (m,n) @ (n,p) -> (m,p); mismatched inner dimensions fail
try:
    np.ones((2, 3)) @ np.ones((2, 3))
except ValueError as err:
    print("ValueError:", err)

ValueError: matmul: Input operand 1 has a mismatch in its core dimension 0, with gufunc signature (n?,k),(k,m?)->(n?,m?) (size 2 is different from 3)


In [45]:
A = np.array([[4.0, 7.0], [2.0, 6.0]])

print(np.linalg.det(A).round(3))          # determinant
print(np.linalg.inv(A).round(3))          # inverse
print((A @ np.linalg.inv(A)).round(10))   # a matrix times its inverse

vals, vecs = np.linalg.eig(np.array([[2.0, 0.0], [0.0, 3.0]]))
print(vals)

coef = np.array([[2.0, 1.0], [1.0, 3.0]])   # 2x +  y = 8
rhs  = np.array([8.0, 13.0])                #  x + 3y = 13
solution = np.linalg.solve(coef, rhs)
print(solution)
print(np.allclose(coef @ solution, rhs))    # always check by substituting back

10.0
[[ 0.6 -0.7]
 [-0.2  0.4]]
[[ 1. -0.]
 [ 0.  1.]]
[2. 3.]
[2.2 3.6]
True


---
## 11. Two Worked Examples

### 11.1 Standardising a dataset

Shift every column to a mean of 0 and scale it to a standard deviation of 1 — the step almost
every machine-learning model expects. Two loops in plain Python; one line here.

In [46]:
rng  = np.random.default_rng(1)
data = rng.integers(30, 100, size=(6, 3))      # 6 students, 3 subjects
print(data)
print("column means:", data.mean(axis=0).round(2))

standard = (data - data.mean(axis=0)) / data.std(axis=0)
print(standard.round(2))

print("new means:", standard.mean(axis=0).round(10))
print("new stds :", standard.std(axis=0).round(10))

[[63 65 82]
 [96 32 40]
 [87 96 47]
 [51 90 59]
 [49 87 47]
 [58 75 68]]
column means: [67.33 74.17 57.17]
[[-0.24 -0.43  1.73]
 [ 1.6  -1.97 -1.19]
 [ 1.1   1.02 -0.71]
 [-0.91  0.74  0.13]
 [-1.03  0.6  -0.71]
 [-0.52  0.04  0.75]]
new means: [ 0. -0.  0.]
new stds : [1. 1. 1.]


### 11.2 An image is a tensor

A colour image *is* an array of shape `(height, width, 3)` holding `uint8` values.

In [47]:
rng   = np.random.default_rng(3)
image = rng.integers(0, 256, size=(4, 4, 3), dtype=np.uint8)
print(image.shape, image.dtype, image.nbytes, "bytes")

grey = image.mean(axis=2)          # average the colour axis away
print(grey.round(1))

print(image[:2, :2, 0])            # the red channel of the top-left corner

(4, 4, 3) uint8 48 bytes
[[210.7 126.7 167.7 165. ]
 [107.  116.7 118.  106.3]
 [ 59.  141.7  57.   15.7]
 [125.3  24.   51.7 146.3]]
[[248 207]
 [  5  60]]


In [48]:
# brightening WRONGLY - uint8 wraps around and bright pixels go dark
wrong = image + 40
print("original red:", image[0, 0, 0], "-> wrong:", wrong[0, 0, 0])

# brightening correctly: widen the type, clip, then narrow again
right = np.clip(image.astype(np.int16) + 40, 0, 255).astype(np.uint8)
print("original red:", image[0, 0, 0], "-> right:", right[0, 0, 0])

original red: 248 -> wrong: 32
original red: 248 -> right: 255


---
# 12. Class Tasks

Complete all five during the lab session and demonstrate them to the instructor before you
leave. **Unless a task says otherwise, no Python `for` loop may appear in your solution** — that
restriction is the whole point of this lab. Print the `shape` and `dtype` of every array you
create, and run **Kernel ▸ Restart & Run All** before you call the instructor.

### C1 — Array Workshop
*creation · attributes · dtypes*

Build each of these **without writing out the values by hand**, and for each print the array,
its `shape`, `ndim`, `size`, `dtype` and `nbytes`:

1. a vector of the even integers strictly between 20 and 50
2. a 3 × 3 matrix of the odd numbers from 1 up to but not including 19
3. ten evenly spaced values from 0 to 2 inclusive
4. a 4 × 4 identity matrix
5. a 2 × 3 × 4 tensor of zeros

Then demonstrate three dtype behaviours, each with a one-line comment explaining it: assigning
`9.9` into an integer array stores `9`; `np.array([120, 125], dtype=np.int8) + 10` wraps around
to negatives; mixing a string into a list of numbers makes the whole array textual. Finish by
comparing the memory used by `list(range(10000))` and `np.arange(10000)` and printing the ratio.

In [49]:
# C1 - Array Workshop
# Name:
# Roll number:

# Your code here

### C2 — The Marks Table
*indexing · slicing · views vs copies · masks*

With `rng = np.random.default_rng(42)`, create a 6 × 4 integer array of marks between 30 and 100
(six students, four subjects). Without any loop, print:

* the third student's whole row
* the second subject's column
* the 3 × 2 block of the first three students and the last two subjects
* every mark of 50 or more
* the marks between 60 and 85
* how many marks are failing
* the array with every failing mark lifted to exactly 50 — **leaving the original unchanged**

Then prove the view-versus-copy rule in code: take a slice of the first two rows, write `999`
into it, and show the original changed; repeat with `.copy()` and show it did not. State in a
comment which of a boolean mask, a basic slice and fancy indexing give a view and which a copy.

In [50]:
# C2 - The Marks Table
# Name:
# Roll number:

# Your code here

### C3 — Reshaping and Broadcasting Drills

Start from `np.arange(24)`. Reshape it to `(4, 6)`, then `(2, 3, 4)`, then `(6, -1)`, printing
each shape. Flatten one with both `ravel` and `flatten` and show in code that only one is a view.
Transpose the `(4, 6)` matrix and confirm the shape.

For the broadcasting half, **predict on paper first**, then verify each pair — writing the result
shape, or the exact `ValueError` message if it is rejected:

`(4,6)` with `(6,)` · `(4,6)` with `(4,)` · `(4,6)` with `(4,1)` · `(3,1)` with `(1,5)`

Repair the failing case with `np.newaxis`. Finish by using broadcasting — and no loop — to
multiply each column of a 4 × 3 marks table by the weights `[0.25, 0.35, 0.40]`, and print each
student's weighted total.

In [51]:
# C3 - Reshaping and Broadcasting Drills
# Name:
# Roll number:

# Your code here

### C4 — Vectorization Contest
*loops against arrays*

For an array of one million floats, write each of the following **twice** — once with a Python
loop, once as a single NumPy expression — timing both with `time.perf_counter()` and printing
the speed-up:

1. `y = 3x² + 2x + 1`
2. the count of elements greater than 0.5
3. the sum of the squares of only the elements above the mean
4. replacing every negative value with zero

Verify with `np.allclose` that each pair of results actually agrees. Present the four speed-ups
in a small printed table, and write two sentences in a comment explaining *why* NumPy is faster —
your explanation must mention the type of the elements and where the loop actually runs.

In [52]:
# C4 - Vectorization Contest
# Name:
# Roll number:

rng = np.random.default_rng(7)
x = rng.standard_normal(1_000_000)

# Your code here

### C5 — A Class Result Sheet
*the whole lab in one program*

With a seeded generator build a marks table for 10 students and 5 subjects (0–100) plus a list of
subject names. Without any loop, compute and print:

* every student's total and average
* every subject's mean, highest and lowest mark
* the standardised table (each subject mean 0, standard deviation 1)
* a letter grade for every mark using `np.where` (nested if needed) — not `if`
* the roll number of the topper, found with `argmax`
* the name of the hardest subject, that is the one with the lowest mean

Then add a 5 % bonus to the weakest subject only, using broadcasting with a weights vector rather
than a loop, and use `np.clip` so no mark exceeds 100. Print the table before and after. Finish
by saving the final table with `np.savetxt`, loading it back with `np.loadtxt`, and showing with
`np.allclose` that the round trip is exact.

In [53]:
# C5 - A Class Result Sheet
# Name:
# Roll number:

subjects = ['Physics', 'Chemistry', 'Maths', 'English', 'Computing']

# Your code here

---
# 13. Home Tasks

These are **not** done in this notebook. Write each in a separate script named
`RollNo_Lab07_Qn.py` (for example `22K-1234_Lab07_Q8.py`), print the shape of every array you
create, seed every random generator, and begin every file with a comment giving your name, roll
number and question number.

1. Create a NumPy vector of all the even integers from 20 to 50, excluding both 20 and 50. Print the vector, its length and its sum.
2. Create a 3 × 3 array of the odd numbers from 1 to 19 (1 included, 19 excluded). Print every element twice — once with `nditer`, once with a single vectorized operation instead of iterating — and explain the difference in a comment.
3. Create a 3 × 3 matrix of even numbers, multiply every element by 4, then multiply the result by a 3 × 3 identity matrix (**not** hard-coded). Show the final result equals the intermediate one and explain why.
4. Generate a 4 × 4 matrix whose entries are drawn at random only from 2, 5, 9 and 10; multiply it by the identity matrix; add the result to any 4 × 4 matrix of odd numbers. Seed the generator.
5. Multiply a 5 × 3 matrix by a 3 × 2 matrix and print the result's shape. Then show what `*` does to two matrices of the same shape, and explain the difference between the two operators.
6. Create a random array of 1000 elements and compute its average, variance and standard deviation. Save all three to a text file created at run time, then read the file back and print it.
7. Create a structured array of student names, heights and classes with appropriate dtypes. Sort by class, and by height where the class is equal. Print before and after.
8. Given a 6 × 6 matrix of random integers from 1 to 100, use slicing and masks — no loop — to print: the four corners; the border elements only; the main diagonal; every even element; and the matrix with every element above 50 replaced by 0.
9. Normalise a 5 × 4 matrix twice: once so every **column** has mean 0 and standard deviation 1, once so every **row** does. Use broadcasting with `keepdims`, and verify by printing the new means and standard deviations.
10. Time the element-wise product of two sequences of 500,000 numbers, once with a Python loop over lists and once with NumPy. Report both times and the speed-up, and confirm agreement with `np.allclose`.
11. Represent a greyscale image as an 8 × 8 `uint8` array. Without any loop produce and print: the image flipped upside down; flipped left to right; rotated 90°; brightened by 50 without wrapping (widen and `np.clip`); and thresholded to pure black and white at 128.
12. Solve `3x + 2y − z = 10`, `2x − 2y + 4z = −2`, `−x + 0.5y − z = 0` with `np.linalg.solve`. Verify by substituting back with `@`, print the determinant and the inverse of the coefficient matrix, and say in a comment what a determinant of zero would have meant.

---
<div style="text-align:center;color:#666"><i>End of Lab 07 — FAST NUCES, Karachi Campus</i></div>